# Track 1 actual translation candidate verification
Read actual before/after predictions and the licensed reference MHR offline on CPU. This checks geometry invariance; it does not establish held-out accuracy.

In [ ]:
import sys,subprocess,json
from pathlib import Path
inputs=Path('/kaggle/input')
models=list(inputs.rglob('mhr_model.pt'));assert len(models)==1,models
before=list(p.parent for p in inputs.rglob('episode_000000.npz') if p.parent.name=='before')
after=list(p.parent for p in inputs.rglob('episode_000000.npz') if p.parent.name=='after')
assert len(before)==len(after)==1,(before,after)
checker=Path('/kaggle/working/check_common_translation.py')
checker.write_text('"""Validate a real translation-filter candidate with the actual reference MHR."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with path.open(\'rb\') as stream:\n        while chunk := stream.read(8 * 1024 * 1024):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--before\', type=Path, required=True)\n    p.add_argument(\'--after\', type=Path, required=True)\n    p.add_argument(\'--mhr-model\', type=Path, required=True)\n    p.add_argument(\'--report\', type=Path, required=True)\n    a = p.parse_args()\n    model_sha = digest(a.mhr_model)\n    if model_sha != \'352e271a6c42729c68554ceaea0c955e866970160c31e35506d782dc0f7377bc\':\n        raise ValueError(\'Use the exact reference MHR model verified by the native exporter\')\n    torch.set_num_threads(2)\n    model = torch.jit.load(str(a.mhr_model), map_location=\'cpu\').eval()\n    flip = torch.tensor([1., -1., -1.])\n    records = []\n    for episode in range(30):\n        stem = f\'episode_{episode:06d}\'\n        before_path, after_path = a.before / (stem + \'.npz\'), a.after / (stem + \'.npz\')\n        with np.load(before_path, allow_pickle=False) as payload:\n            before = {k: payload[k].copy() for k in payload.files}\n        with np.load(after_path, allow_pickle=False) as payload:\n            after = {k: payload[k].copy() for k in payload.files}\n        if set(before) != set(after):\n            raise ValueError(\'Unexpected candidate parameter fields\')\n        for key in set(before) - {\'pose\', \'object_translation\'}:\n            np.testing.assert_array_equal(before[key], after[key])\n        np.testing.assert_array_equal(before[\'pose\'][:, 3:], after[\'pose\'][:, 3:])\n        np.testing.assert_array_equal(before[\'pose\'][0], after[\'pose\'][0])\n        np.testing.assert_array_equal(before[\'object_translation\'][0], after[\'object_translation\'][0])\n        length = len(before[\'pose\'])\n        delta = after[\'object_translation\'].astype(np.float64) - before[\'object_translation\']\n        indices = sorted({0, length // 2, length - 1, int(np.linalg.norm(delta, axis=1).argmax())})\n        identity = torch.tensor(np.repeat(before[\'shape\'][None], len(indices), axis=0), dtype=torch.float32)\n        expression = torch.zeros(len(indices), 72)\n\n        def decode(values):\n            parameters = np.concatenate([values[\'pose\'][indices],\n                                         np.repeat(values[\'scales\'][None], len(indices), axis=0)], axis=1)\n            with torch.inference_mode():\n                vertices, _ = model(identity, torch.tensor(parameters, dtype=torch.float32), expression, True)\n            return vertices * flip / 100.\n\n        actual_delta = (decode(after) - decode(before)).numpy()\n        error = np.linalg.norm(actual_delta - delta[indices, None, :], axis=-1)\n        maximum = float(error.max())\n        if not np.isfinite(error).all() or maximum > 0.00005:\n            raise ValueError(f\'{stem}: human/object shift differs by {maximum} m\')\n        before_mesh = a.before / (stem + \'_object.glb\')\n        after_mesh = a.after / (stem + \'_object.glb\')\n        if digest(before_mesh) != digest(after_mesh):\n            raise ValueError(\'Candidate mesh changed\')\n        records.append({\'episode\': episode, \'frames\': length, \'reference_checked_frames\': indices,\n                        \'maximum_reference_shift_error_m\': maximum,\n                        \'source_npz_sha256\': digest(before_path),\n                        \'output_npz_sha256\': digest(after_path), \'mesh_sha256\': digest(after_mesh)})\n        print(\'REFERENCE_TRANSLATION_CHECK\', episode, maximum, flush=True)\n    report = {\'passed\': True, \'mhr_model_sha256\': model_sha,\n              \'checker_script_sha256\': digest(Path(__file__)), \'torch_version\': torch.__version__,\n              \'first_frame_and_nontranslation_parameters_unchanged\': True,\n              \'geometry_claim\': \'Shared human/object shifts verified against actual MHR on sampled real frames; all pose controls except translation are exactly unchanged.\',\n              \'held_out_accuracy_improvement_established\': False, \'episodes\': records}\n    a.report.parent.mkdir(parents=True, exist_ok=True)\n    a.report.write_text(json.dumps(report, indent=2) + \'\\n\')\n\n\nif __name__ == \'__main__\':\n    main()\n')
subprocess.run([sys.executable,str(checker),'--before',str(before[0]),'--after',str(after[0]),
 '--mhr-model',str(models[0]),'--report','/kaggle/working/common_translation_decoder_check.json'],check=True)
